---
title: Omeka S API Basics
---

This sections illustrates the basic functions of the Omeka S API.
This exploration uses python to query the API, but other tools can be similarly effective.

:::{seealso} Omeka S API Documentation
For reference, the Omeka S documentation can be accessed at https://omeka.org/s/docs/developer/api/rest_api/.
See [this post](https://forum.omeka.org/t/example-api-usage-using-curl/8083) on using `curl` to query the API.
:::

Any public objects can be accessed or viewed in the API without credentials,
but private data and write or modify options require an authentication process.
As the Omeka S documentation explains, "[t]he API permits anonymous access to public resources (i.e., reading non-private data). To perform actions or view data that only logged-in users can access, you must authenticate your requests."
The following examples authenticate requests using access keys stored in a file named `../omeka-credentials.json`.
To create this file for your own site, copy `../data/omeka-credentials-template.json` to the repository root as `omeka-credentials.json` and fill in your keys:

```{literalinclude} ../data/omeka-credentials-template.json
:label: omekas-api-credential-template
```

:::{warning} Do not commit or upload access credentials
The access key and secret allow a user to make significant changes to your site content, which could include adding, modifying, and deleting information.
To secure your site, these kinds of credentials should never be shared publicly, so
never commit the filled-in file.
:::

## Setup

This section will use the `json` library to parse JSON data and `requests` to manage HTTP interactions.

In [ ]:
import requests
import json

### Get Credentials

Many operations do not require access credentials. Content that is publicly displayed can be accessed, read, and often even downloaded. Actions like adding, modifying, or deleting content, however, do require credentials. While these codes should not be uploaded publicly, the following function can be used to load and reuse credentials for your site while not uploading them to the open web.

In [ ]:
def get_credentials(credential_file_path):
    '''Retrieve Omeka S API credentials from another file. 
    That file must be a JSON file.'''

    with open(credential_file_path, 'r') as credentials:
        keys = json.load(credentials)
    
    return keys['key_identity'], keys['key_credential']

In [ ]:
# retrieve API credentials

credential_file_path = '../omeka-credentials.json'

key_identity, key_credential = get_credentials(credential_file_path)

Now it's possible for your scripts to authenticate requests to the Omeka S API,
allowing for posting new data and modifying information in your site's databases.

The API interface allows a script or other programmatic tool
to work with the data and items in your Omeka S sites.
The next section illustrates a basic APi call that requests
descriptive metadata from the Omeka S site.

## A Basic API Call: Get a List of Items

To begin, it may be useful to get an inventory of current items available in your Omeka S site.
To do this, the API's `items` action is a good option.

### Set Up the URL

The next few cells demonstrate how to build the URL for the request.
First, establish the base URL (your site's main address), then append `/api` as an endpoint.

In [ ]:
# establish API location/endpoints
siteUrl = 'http://jajohnst.si676.si.umich.edu/omekas_2026' # if you replicate this example, provide the URI for your site

endpoint = '/api'

action = '/items'

Then, assemble the URI.

In [ ]:
r_url = siteUrl + endpoint + action

print(r_url)

### Set Up Parameters

Although credentials are not required for basic API usage,
many requests will require them.
The next cell shows how to set up parameters that add the credentials to the request URL,
return a "pretty printed" JSON response, and specify that you want
the JSON-LD serialization in the response.

In [ ]:
parameters = {
    'key_credential': key_credential,
    'key_identity':   key_identity,
    'pretty_print':   1,
    'format':         'jsonld'
}

### Send the Request

Now that the request information is assembled, send the request.

In [ ]:
r = requests.get(r_url, params=parameters)

print(r_url) # r.url would include the API keys
print(r.status_code)

### Examine the Response

Now take a look at the response. The following cells use the built-in `json` parser that comes with the `requests` library.

In [ ]:
len(r.json())

In [ ]:
for element in r.json():
    print(element)

In [ ]:
for element in r.json():
    print(element['o:id'],':',element['o:title'])

## Get the information about just one item

In [ ]:
item_id = r.json()[0]['o:id']
print(item_id)

Construct a new URI:

In [ ]:
r_url = siteUrl + endpoint + action + '/' + str(item_id)

Make the request:

In [ ]:
r = requests.get(r_url, params=parameters)

print(r_url) # r.url would include the API keys
print(r.status_code)

In [ ]:
r.json()

Save the item information locally as a json file

In [ ]:
with open(f'../output/item-{item_id}-metadata.json', 'w') as f:
    f.write(json.dumps(r.json(), indent=2))
    print('wrote metadata to local file')

## Retrieve multiple items, using a feeder list

Based on the full inventory from the `api/items` action, retrieve individual items:

In [ ]:
# get all items
action = '/items'

r_url = siteUrl + endpoint + action

r = requests.get(r_url, params=parameters)

print(r_url) # r.url would include the API keys
print(r.status_code)

In [ ]:
# record each item id in a list
item_list = list()

for item in r.json():
    item_list.append(item['o:id'])

print(item_list)

In [ ]:
# retrieve the individual item records and save to a file

count = 0

for item in item_list:
    r_url = siteUrl + endpoint + action + '/' + str(item)
    print(f'requesting {r_url}')
    r = requests.get(r_url, params=parameters)
    fname = 'item-' + str(item) + '-metadata.json'
    with open(f'../output/{fname}', 'w') as f:
        f.write(json.dumps(r.json(), indent=2))
    print(f'saved to file {fname}')
    count += 1

print(f'\nwrote {count} new files')

## Getting the "Context"

You may have noted each of the keys in the Omeka S JSON responses have a namespace.
This is because this response is JSON-LD, or JSON linked data. The `@context` tag refers to the API endpoint where these are defined.
In this case, the endpoint is `api-context`. 

Request the context:

In [ ]:
r_url = siteUrl + '/api-context'

r = requests.get(r_url, params=parameters)

with open('../output/omeka-s-api-context.json', 'w') as f:
    f.write(json.dumps(r.json(), indent=2))

In [ ]:
r_url

In [ ]:
r.json()